# Gaz-wa: eksploracja danych

Ten notebook jest miejscem do bezpiecznej pracy lokalnej. Zmien `DATA_PATH`, `TIMESTAMP_COL`, `TARGET_COL` i opcjonalnie `TEMPERATURE_COL` pod swoje dane.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from gaz_wa.config import DataSchema
from gaz_wa.data_loading import load_table, normalize_columns
from gaz_wa.eda import describe_frame
from gaz_wa.features import build_feature_frame
from gaz_wa.validation import prepare_time_series, validate_raw_frame

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

## Konfiguracja pliku

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "sample_gas_consumption.csv"
TIMESTAMP_COL = "timestamp"
TARGET_COL = "gas_consumption"
TEMPERATURE_COL = "outside_temperature"  # ustaw None, jesli nie masz temperatury
NORMALIZE_COLUMNS = False

df = load_table(DATA_PATH)
if NORMALIZE_COLUMNS:
    df = normalize_columns(df)

schema = DataSchema(timestamp_col=TIMESTAMP_COL, target_col=TARGET_COL)
df.head()

## Walidacja i podstawowy opis

In [ ]:
report = validate_raw_frame(df, schema)
print(f"Wiersze: {report.rows}, kolumny: {report.columns}")
for issue in report.issues:
    print(f"{issue.severity.upper()}: {issue.column or '-'}: {issue.message}")

In [ ]:
tables = describe_frame(df)
tables["dtypes"]

In [ ]:
tables["missing"]

## Szereg czasowy i cechy

In [ ]:
ts = prepare_time_series(df, schema)
features = build_feature_frame(ts, target_col=TARGET_COL, temperature_col=TEMPERATURE_COL)
features.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
features[TARGET_COL].plot(ax=ax)
ax.set_title("Zuzycie gazu w czasie")
ax.set_xlabel("czas")
ax.set_ylabel(TARGET_COL);

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(features[TARGET_COL].dropna(), kde=True, ax=ax)
ax.set_title("Rozklad zuzycia gazu");

In [ ]:
hourly_profile = features.groupby("hour")[TARGET_COL].agg(["mean", "median", "min", "max"])
hourly_profile

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
hourly_profile[["mean", "median"]].plot(ax=ax)
ax.set_title("Profil godzinowy")
ax.set_xlabel("godzina")
ax.set_ylabel(TARGET_COL);

In [ ]:
numeric = features.select_dtypes(include="number")
corr = numeric.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(max(8, len(corr) * 0.6), max(6, len(corr) * 0.5)))
sns.heatmap(corr, cmap="vlag", center=0, ax=ax)
ax.set_title("Korelacje");

In [ ]:
if TEMPERATURE_COL and TEMPERATURE_COL in features.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=features, x=TEMPERATURE_COL, y=TARGET_COL, ax=ax)
    ax.set_title("Zuzycie gazu vs temperatura")